<a href="https://colab.research.google.com/github/rohith154-vsr/ML-2-LAB/blob/main/LP_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install networkx matplotlib pandas seaborn -q

import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# First-order rule learning: learning scholarship eligibility from relational facts
# Background knowledge: sets of students satisfying each predicate
facts = {
    "HighGPA": {("Alice",), ("Bob",), ("David",), ("Emma",), ("Grace",), ("Harry",)},
    "GoodAttendance": {("Alice",), ("Carol",), ("David",), ("Emma",), ("Frank",), ("Grace",)},
    "Extracurricular": {("Alice",), ("Bob",), ("Emma",), ("Frank",), ("Grace",)},
    "LowIncome": {("Alice",), ("David",), ("Emma",), ("Grace",), ("Ivy",)},
    "FullTime": {("Alice",), ("Bob",), ("Carol",), ("David",), ("Emma",), ("Frank",), ("Grace",), ("Harry",), ("Ivy",)}
}

# training examples: positive = eligible for scholarship (True), negative = ineligible (False)
examples = [
    ("Alice", None, True),
    ("David", None, True),
    ("Emma", None, True),
    ("Grace", None, True),
    ("Bob", None, False),
    ("Carol", None, False),
    ("Frank", None, False),
    ("Harry", None, False),
    ("Ivy", None, False),
]

# candidate first-order literals that can be added to the rule body
candidate_literals = {
    "HighGPA(x)": lambda x, y: (x,) in facts["HighGPA"],
    "GoodAttendance(x)": lambda x, y: (x,) in facts["GoodAttendance"],
    "Extracurricular(x)": lambda x, y: (x,) in facts["Extracurricular"],
    "LowIncome(x)": lambda x, y: (x,) in facts["LowIncome"],
    "FullTime(x)": lambda x, y: (x,) in facts["FullTime"],
}

# a rule covers an example only if ALL candidate literals in its body hold true
def evaluate_rule(literal_names, examples):
    pos, neg, covered = 0, 0, []
    for x, y, label in examples:
        satisfied = all(candidate_literals[lit](x, y) for lit in literal_names)
        if satisfied:
            covered.append((x, y, label))
            if label:
                pos += 1
            else:
                neg += 1
    return pos, neg, covered

# Learn-One-Rule algorithm:
# Starts with the most general rule (empty body covering all instances)
# Greedily adds the literal that maximizes positive instance purity (pos / (pos + neg))
# Stops as soon as the rule covers zero negative examples
def learn_one_rule(examples, candidate_literals):
    rule_body = []
    history = []
    round_num = 0

    while True:
        pos, neg, covered = evaluate_rule(rule_body, examples)
        history.append({
            "round": round_num,
            "rule": " AND ".join(rule_body) if rule_body else "(most general)",
            "pos_covered": pos,
            "neg_covered": neg,
        })
        if neg == 0:
            break # pure rule achieved (covers no negatives)

        # evaluate each candidate literal extension
        scores = {}
        for lit in candidate_literals:
            if lit in rule_body:
                continue
            trial = rule_body + [lit]
            p, n, _ = evaluate_rule(trial, examples)
            scores[lit] = p / (p + n) if (p + n) > 0 else 0.0

        if not scores:
            break

        best_lit = max(scores, key=scores.get)
        rule_body.append(best_lit)
        round_num += 1

    return rule_body, history

rule, history = learn_one_rule(examples, candidate_literals)

print("Learned Rule:")
print(" Eligible(x) <-", " AND ".join(rule) if rule else "(no conditions)")

print("
Search History:")
for h in history:
    print(f"  Round {h['round']}: {h['rule']}")
    print(f"    Positives covered: {h['pos_covered']}, Negatives covered: {h['neg_covered']}")

# visualize search progression from general to specialized rule as a tree
G = nx.DiGraph()
prev_label = None

for h in history:
    label = h["rule"]
    node_label = f"{label}
P={h['pos_covered']} N={h['neg_covered']}"
    G.add_node(label, display=node_label)
    if prev_label is not None:
        G.add_edge(prev_label, label)
    prev_label = label

pos_layout = nx.spring_layout(G, seed=7)
labels = {n: data["display"] for n, data in G.nodes(data=True)}

plt.figure(figsize=(10, 5))
nx.draw(G, pos_layout, labels=labels, with_labels=True,
        node_color="lightblue", node_size=3200,
        font_size=8, arrows=True, edge_color="gray")
plt.title("Learn-One-Rule Search Tree (General -> Specific)")
plt.tight_layout()
plt.show()